In [3]:
import json
import random

# Your provided dataset

with open("full_list.json", "r") as f:
    data = json.load(f)
def split_train_val_test(dataset, val_ratio=0.15, test_ratio=0.15, seed=42):
    # 1. Group items by the EXACT room name 
    grouped_rooms = {}
    for item in dataset:
        room_name = item.split(" ")[0] 
        
        if room_name not in grouped_rooms:
            grouped_rooms[room_name] = []
        grouped_rooms[room_name].append(item)

    # 2. Shuffle the unique room names
    unique_rooms = list(grouped_rooms.keys())
    random.seed(seed)
    random.shuffle(unique_rooms)

    # 3. Calculate split indices
    total_rooms = len(unique_rooms)
    num_test = max(1, int(total_rooms * test_ratio))
    num_val = max(1, int(total_rooms * val_ratio))
    
    test_rooms = unique_rooms[:num_test]
    val_rooms = unique_rooms[num_test : num_test + num_val]
    train_rooms = unique_rooms[num_test + num_val:]

    # --- Print exact room allocations ---
    print("========================================")
    print(f"🏢 TOTAL UNIQUE ROOMS: {total_rooms}")
    print("========================================\n")
    
    print(f"🟢 TRAIN SET ({len(train_rooms)} rooms):")
    for room in sorted(train_rooms):
        print(f"  - {room} (Contains {len(grouped_rooms[room])} entries)")

    print(f"\n🟡 VALIDATION SET ({len(val_rooms)} rooms):")
    for room in sorted(val_rooms):
        print(f"  - {room} (Contains {len(grouped_rooms[room])} entries)")
        
    print(f"\n🔵 TEST SET ({len(test_rooms)} rooms):")
    for room in sorted(test_rooms):
        print(f"  - {room} (Contains {len(grouped_rooms[room])} entries)")
    print("\n========================================")

    # 4. Populate the final lists
    train_list = []
    for room in train_rooms:
        train_list.extend(grouped_rooms[room])
        
    val_list = []
    for room in val_rooms:
        val_list.extend(grouped_rooms[room])
        
    test_list = []
    for room in test_rooms:
        test_list.extend(grouped_rooms[room])

    # --- Strict Validation Check across ALL THREE sets ---
    train_set = set([item.split(" ")[0] for item in train_list])
    val_set = set([item.split(" ")[0] for item in val_list])
    test_set = set([item.split(" ")[0] for item in test_list])
    
    leakage_train_val = train_set.intersection(val_set)
    leakage_train_test = train_set.intersection(test_set)
    leakage_val_test = val_set.intersection(test_set)
    
    if any([leakage_train_val, leakage_train_test, leakage_val_test]):
        print(f"❌ VALIDATION FAILED! Data leakage detected.")
        if leakage_train_val: print(f"  Leakage between Train & Val: {leakage_train_val}")
        if leakage_train_test: print(f"  Leakage between Train & Test: {leakage_train_test}")
        if leakage_val_test: print(f"  Leakage between Val & Test: {leakage_val_test}")
    else:
        print("✅ VALIDATION PASSED: 0 data leakage across Train, Val, and Test sets.")

    # 5. Save to JSON files
    with open("train_list.json", "w") as f:
        json.dump(train_list, f, indent=4)
        
    with open("val_list.json", "w") as f:
        json.dump(val_list, f, indent=4)
        
    with open("test_list.json", "w") as f:
        json.dump(test_list, f, indent=4)
        
    print("\n💾 Successfully saved 'train_list.json', 'val_list.json', and 'test_list.json'!")

if __name__ == "__main__":
    # You can adjust the ratios here. 0.15 test and 0.15 val leaves 0.70 (70%) for train.
    split_train_val_test(data, val_ratio=0.1, test_ratio=0.2)

🏢 TOTAL UNIQUE ROOMS: 27

🟢 TRAIN SET (20 rooms):
  - frl_apartment_0_000 (Contains 100 entries)
  - frl_apartment_2_000 (Contains 100 entries)
  - frl_apartment_3_000 (Contains 100 entries)
  - frl_apartment_4_000 (Contains 100 entries)
  - frl_apartment_5_000 (Contains 100 entries)
  - large_apartment_0_000 (Contains 100 entries)
  - large_apartment_0_002 (Contains 100 entries)
  - large_apartment_0_003 (Contains 100 entries)
  - large_apartment_0_005 (Contains 100 entries)
  - large_apartment_0_006 (Contains 100 entries)
  - large_apartment_0_007 (Contains 100 entries)
  - large_apartment_1_000 (Contains 100 entries)
  - large_apartment_1_001 (Contains 100 entries)
  - large_apartment_2_000 (Contains 100 entries)
  - office_0_000 (Contains 100 entries)
  - office_2_000 (Contains 100 entries)
  - office_4_000 (Contains 100 entries)
  - room_0_000 (Contains 100 entries)
  - room_1_000 (Contains 100 entries)
  - room_2_000 (Contains 100 entries)

🟡 VALIDATION SET (2 rooms):
  - large_a